# iChart-lite: GitHub Public Notebook

This notebook is the GitHub-ready public version of the iChart-lite implementation.

It assumes the following repository structure:

```text
ichart-lite/
├── iChart.ipynb
├── models/
│   ├── yolov8s.yaml
│   └── yolov8s-best-82.pt
└── data/
    ├── data_82.yaml
    ├── train.txt
    ├── val.txt
    ├── images.json
    ├── images/
    └── labels/
```

The sample repository can include a reduced subset of images and labels for demonstration.
To reproduce the full paper results, the complete dataset and model files should be prepared in the same structure.


## 1. Environment Setup


In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

!sudo apt install -y tesseract-ocr
!pip install -q gtts Levenshtein pytesseract ultralytics

import json
import os
import re
import time
from glob import glob
from typing import TypedDict

import cv2
import Levenshtein
import numpy as np
import pytesseract
from gtts import gTTS
from IPython.display import Audio
from PIL import Image
from ultralytics import YOLO


## 2. Repository Path Configuration


In [ ]:
from pathlib import Path

# Repository root inside Colab.
# Change GREENEYE_REPO_ROOT / ICHART_REPO_ROOT only if your folder name is different.
REPO_ROOT = Path(os.environ.get("ICHART_REPO_ROOT", "/content/ichart-lite"))

# Data and model paths
DATA_DIR = REPO_ROOT / "data"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
ANSWERS_JSON = DATA_DIR / "images.json"
DATA_YAML = DATA_DIR / "data_82.yaml"

MODEL_CONFIG = REPO_ROOT / "models" / "yolov8s.yaml"
MODEL_WEIGHTS = Path(os.environ.get("ICHART_WEIGHTS", str(REPO_ROOT / "models" / "yolov8s-best-82.pt")))

# Main runtime settings
OFFSET = 7
PREDICT_CONF = 0.3
EVAL_CONF = 0.5
PYTESSERACT_CONFIG = "--psm 6"

print("REPO_ROOT    :", REPO_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("IMAGES_DIR   :", IMAGES_DIR)
print("LABELS_DIR   :", LABELS_DIR)
print("ANSWERS_JSON :", ANSWERS_JSON)
print("DATA_YAML    :", DATA_YAML)
print("MODEL_CONFIG :", MODEL_CONFIG)
print("MODEL_WEIGHTS:", MODEL_WEIGHTS)


## 3. Load YOLOv8 Model


In [ ]:
# Load the YOLOv8 model configuration and pretrained weights.
model = YOLO(str(MODEL_CONFIG))
model = model.load(str(MODEL_WEIGHTS))

# Set readable class names for plotting and debugging.
names = ["bar", "x", "y"]
for i, name in enumerate(names):
    model.names[i] = name

model.info()


## 4. Core Extraction Logic


In [ ]:
# Main extraction pipeline

Y_LABEL_REGEX = re.compile(r"(\D*)(\d+(?:\.\d+)?)(\D*)")

### YOLO helper logicv8 detection bounding box
###
### Helper class for bounding-box geometry
class Detection:
  xyxy: tuple[int, int, int, int] # top-left x, y and bottom-right x, y

  def __init__(self, box):
    self.xyxy = (
      int(box.xyxy[0][0]),
      int(box.xyxy[0][1]),
      int(box.xyxy[0][2]),
      int(box.xyxy[0][3])
    )

  def xl(self) -> int:
    return self.xyxy[0]

  def xr(self) -> int:
    return self.xyxy[2]

  def yb(self) -> int:
    return self.xyxy[3]

  def yt(self) -> int:
    return self.xyxy[1]

  def center(self) -> tuple[int, int]:
    return ((self.xl() + self.xr()) // 2, (self.yt() + self.yb()) // 2)

  def width(self) -> int:
    return self.xr() - self.xl()

  def height(self) -> int:
    return -(self.yt() - self.yb())

  def area(self) -> int:
    return self.width() * self.height()

### YOLO helper logicv8 detection bounding box (X/Y 라벨 특화)
###
### Reads the OCR text directly from the detected label region
class LabelDetection(Detection):
  label: str

  def __init__(self, box, orig_image):
    super().__init__(box)

    detection_image = orig_image[self.xyxy[1]:self.xyxy[3], self.xyxy[0]:self.xyxy[2]]
    detection_image = cv2.cvtColor(detection_image, cv2.COLOR_BGR2GRAY)
    label = pytesseract.image_to_string(detection_image, config=PYTESSERACT_CONFIG).strip()
    self.label = label    \
      .replace("\n", " ") \
      .replace(",", "")   \
      .replace(".", "")   \
      .replace("‘", "")

### YOLOv8 detection bounding box (Y-axis label specific)
###
### Post-process the label string returned by LabelDetection for Y-axis labels:
### 1. Replace 'O'/'o' with '0' because OCR often misreads zero as a letter.
### 2. Use Y_LABEL_REGEX to extract Y-label prefixes and suffixes
###    (e.g., "$10", "10th").
class YLabelDetection(LabelDetection):
  value: float
  prefix: str
  suffix: str

  def __init__(self, box, orig_image):
    super().__init__(box, orig_image)

    # big O and small o to zero (0)
    self.label = self.label.replace("O", "0").replace("o", "0")

    regex_result = Y_LABEL_REGEX.search(self.label)
    if regex_result is None:
      self.value = -1
      self.prefix = "False"
      self.suffix = "False"
    else:
      self.value = float(regex_result.groups()[1])
      self.prefix = regex_result.groups()[0]
      self.suffix = regex_result.groups()[2]

class Detections(TypedDict):
  bar: list[Detection]
  x_label: list[LabelDetection]
  y_label: list[YLabelDetection]

### Outlier removal using IQR and offset
def remove_outlier(l: list, key=lambda v: v, offset=OFFSET) -> list:
  q1 = np.quantile([key(v) for v in l], 0.25)
  q3 = np.quantile([key(v) for v in l], 0.75)
  iqr = q3 - q1

  return list(filter(
      lambda v: (key(v) <= q3 + 1.5*iqr + offset) and (key(v) >= q1 - 1.5*iqr - offset),
      l
  ))

### Main function
###
### Extract chart data from an image by:
### 1. Removing outliers
### 2. Matching bars with X-axis labels
### 3. Estimating the prefix and suffix of Y-axis labels
def extract_data(result, image) -> tuple[list[str], list[float], str, str, float, float] | None:
  detections: Detections = { "bar": [], "x_label": [], "y_label": [] }

  for box in result.boxes:
    name = model.names[int(box.cls)]
    if name == "bar":
      detections["bar"].append(Detection(box))
    elif name == "x":
      detection = LabelDetection(box, image)
      if detection is not None:
        detections["x_label"].append(detection)
    elif name == "y":
      detection = YLabelDetection(box, image)
      if detection is not None:
        detections["y_label"].append(detection)

  ### remove outlier - bar
  detections["bar"] = remove_outlier(detections["bar"], lambda v: v.yb(), 10)
  detections["bar"].sort(key=lambda v: v.center()[0])

  ### remove outlier - X label
  detections["x_label"] = remove_outlier(detections["x_label"], lambda v: v.center()[1])
  detections["x_label"].sort(key=lambda v: v.center()[0])

  ### remove outlier - Y label
  detections["y_label"] = list(filter(lambda v: v.prefix != "False", detections["y_label"]))
  detections["y_label"].sort(key=lambda v: v.center()[1])
  detections["y_label"].reverse()

  ### Error handling: if fewer than two Y-axis labels are detected,
  ### treat the image as an invalid chart and return None
  if len(detections["y_label"]) < 2:
    return None

  y_bottom = detections["y_label"][0]
  y_top = detections["y_label"][-1]

  def get_bar_value(bar_detection):
    ratio = (y_top.value - y_bottom.value)/-(y_top.center()[1] - y_bottom.center()[1])

    return ratio*(bar_detection.height() - (-(y_bottom.center()[1] - bar_detection.yb()))) + y_bottom.value

  ### Estimate the number of bar–X label pairs
  ### from the detected bars and X-axis labels.
  ### The larger count between the two is used
  ### as the estimated number of pairs.
  bar_count_estimate = max(len(detections["bar"]), len(detections["x_label"]))

  ### Estimate bar–X label pairs based on their X-axis coordinates
  ###
  ### Basic algorithm:
  ### 1. Start with the first detected bar and the first detected X-axis label.
  ### 2.1 If the first bar and first X-label correspond to each other,
  ###     add them to bar_values and x_labels.
  ### 3.1 Move to the second detected bar and the second detected X-label.
  ###
  ### 2.2 If they do not correspond, it means either the first detected bar
  ###     or the first detected X-label is not actually the first pair.
  ###     Compare their X-axis coordinates to determine which one comes first,
  ###     and insert a default value for the missing element.
  ###
  ### 3.2 Depending on the result of (2.2), proceed with either:
  ###     {second detected bar, first detected X-label}
  ###     or
  ###     {first detected bar, second detected X-label}.
  ###
  ### 4. Repeat the process until all detected elements are processed.
  bar_values = []
  x_labels = []
  bar_i = 0
  x_label_i = 0
  while True:
    ### stop condition
    if len(bar_values) == len(x_labels) and len(bar_values) == bar_count_estimate:
      break

    if len(detections["bar"]) <= bar_i:
      bar_values.append(y_bottom.value) # placeholder

      x_labels.append(detections["x_label"][x_label_i].label)
      x_label_i += 1

      continue

    if len(detections["x_label"]) <= x_label_i:
      # bar_values.append(detections["bar"][bar_i].height()/y_height*(y_top.value - y_bottom.value) + y_bottom.value)
      bar_values.append(get_bar_value(detections["bar"][bar_i]))
      bar_i += 1

      x_labels.append("[Unknown]") # placeholder

      continue

    if (detections["x_label"][x_label_i].center()[0] > detections["bar"][bar_i].xl() and \
        detections["x_label"][x_label_i].center()[0] < detections["bar"][bar_i].xr()) or \
       (detections["bar"][bar_i].center()[0] > detections["x_label"][x_label_i].xl() and
        detections["bar"][bar_i].center()[0] < detections["x_label"][x_label_i].xr()):
      # add bar value
      # bar_values.append(detections["bar"][bar_i].height()/y_height*(y_top.value - y_bottom.value) + y_bottom.value)
      bar_values.append(get_bar_value(detections["bar"][bar_i]))
      bar_i += 1

      # add x label
      x_labels.append(detections["x_label"][x_label_i].label)
      x_label_i += 1
    else:
      # bar is missing
      if detections["x_label"][x_label_i].center()[0] < detections["bar"][bar_i].center()[0]:
        bar_values.append(y_bottom.value) # placeholder

        # add x label
        x_labels.append(detections["x_label"][x_label_i].label)
        x_label_i += 1
      # x_label is missing
      else:
        # add bar value
        # bar_values.append(detections["bar"][bar_i].height()/y_height*(y_top.value - y_bottom.value) + y_bottom.value)
        bar_values.append(get_bar_value(detections["bar"][bar_i]))
        bar_i += 1

        x_labels.append("[Unknown]") # placeholder

  ### Estimate the prefix and suffix of Y-axis labels
  ### Choose the longest prefix and suffix among all detected
  ### Y-axis labels as the final format.
  prefix = max(y_bottom, y_top, key=lambda v: v.prefix).prefix
  suffix = max(y_bottom, y_top, key=lambda v: v.suffix).suffix

  return (x_labels, bar_values, prefix, suffix, y_bottom.value, y_top.value)

def create_description(x_labels: list[str], bar_values: list[float], prefix: str, suffix: str) -> str:
  s = ""

  for x_label, bar_value in zip(x_labels, bar_values):
    bar_value_str = f"{prefix}{bar_value:.2f}{suffix}"
    s += f"The value of {x_label} is {bar_value_str}.\n"

  return s

## 5. Single-Image Demo


In [ ]:
# Select one sample image for a quick demo.
sample_candidates = sorted(glob(str(IMAGES_DIR / "*.jpg")) + glob(str(IMAGES_DIR / "*.JPG")) + glob(str(IMAGES_DIR / "*.png")))

if not sample_candidates:
    raise FileNotFoundError(f"No sample images found in {IMAGES_DIR}")

image_path = sample_candidates[0]
print("Demo image:", image_path)

image = cv2.imread(image_path)

# Step 1: object detection
predict_start = time.perf_counter()
results = model.predict(image, conf=PREDICT_CONF, imgsz=640, verbose=False)
predict_time = time.perf_counter() - predict_start

# Step 2: chart data extraction
extract_start = time.perf_counter()
data = extract_data(results[0], image)
extract_time = time.perf_counter() - extract_start

print(f"Prediction time: {predict_time:.3f} s")
print(f"Extraction time: {extract_time:.3f} s")

if data is not None:
    x_labels, bar_values, prefix, suffix, yb_value, yt_value = data
    description = create_description(x_labels, bar_values, prefix, suffix)
    print(description)
else:
    print("Extraction failed for this sample image.")


## 6. Evaluation on the Sample Metadata


In [ ]:
from itertools import zip_longest

def levenshtein_distance_and_error_rate(g, p):
  len_max = max(len(g), len(p))
  distance = Levenshtein.distance(g, p)
  error_rate = distance / len_max

  return distance, error_rate

def hamming_distance_and_error_rate(g, p):
  len_max = max(len(g), len(p))
  distance = sum(c1 != c2 for c1, c2 in zip_longest(g, p))
  error_rate = distance / len_max

  return distance, error_rate

json_path = str(ANSWERS_JSON)

with open(json_path, "r") as f:
  answers = json.load(f)

label_count = 0
lv_distance_sum = 0
lv_error_rate_sum = 0
lv_incorrect_sum = 0
lv_correct_sum = 0
hm_distance_sum = 0
hm_error_rate_sum = 0
hm_incorrect_sum = 0
hm_correct_sum = 0

for i, answer in enumerate(answers):
  image_path = str(IMAGES_DIR / answer["name"])
  image_name = os.path.basename(image_path)
  image = cv2.imread(image_path)
  print(f"[{i + 1}/{len(answers)}]: {image_name}")

  #results = model.predict(image, conf=CONFIDENCE, imgsz=640, verbose=False)
  results = model.predict(image, conf=0.5, imgsz=640, verbose=False)
  annotated_image = results[0].plot()
    # cv2_imshow(image)

  data = extract_data(results[0], image)

  if data is not None:
    x_labels, bar_values, prefix, suffix, yb_value, yt_value = data

    lv_distances = []
    lv_error_rates = []
    hm_distances = []
    hm_error_rates = []
    for label_answer, label_guess in zip(answer["x_labels"], x_labels):
      if label_guess == "[Unknown]":
        label_guess = ""

      len_max = max(len(label_answer), len(label_guess))

      lv_distance, lv_error_rate = levenshtein_distance_and_error_rate(label_answer, label_guess)
      lv_incorrect = lv_distance
      lv_correct = len_max - lv_incorrect

      hm_distance, hm_error_rate = hamming_distance_and_error_rate(label_answer, label_guess)
      hm_incorrect = hm_distance
      hm_correct = len_max - hm_incorrect

      lv_distance_sum += lv_distance
      lv_error_rate_sum += lv_error_rate
      lv_incorrect_sum += lv_incorrect
      lv_correct_sum += lv_correct
      hm_distance_sum += hm_distance
      hm_error_rate_sum += hm_error_rate
      hm_incorrect_sum += hm_incorrect
      hm_correct_sum += hm_correct
      label_count += 1

      lv_distances.append(lv_distance)
      lv_error_rates.append(lv_error_rate)
      hm_distances.append(hm_distance)
      hm_error_rates.append(hm_error_rate)

    description = create_description(x_labels, bar_values, prefix, suffix)

    print(f"[yb, yt]: [{yb_value}, {yt_value}]")
    print()
    print(lv_distances)
    print(lv_error_rates)
    print(f"lv distance: {sum(lv_distances) / len(lv_distances)}")
    print(f"lv distance so far: {lv_distance_sum / label_count}")
    print(f"lv error rate: {sum(lv_error_rates) / len(lv_error_rates)}")
    print(f"lv error rate so far: {lv_error_rate_sum / label_count}")
    print()
    print(hm_distances)
    print(hm_error_rates)
    print(f"hm distance: {sum(hm_distances) / len(hm_distances)}")
    print(f"hm distance so far: {hm_distance_sum / label_count}")
    print(f"hm error rate: {sum(hm_error_rates) / len(hm_error_rates)}")
    print(f"hm error rate so far: {hm_error_rate_sum / label_count}")
    print()
    print(description)
  else:
    print("Data extraction failed")

  print()

print(f"lv distance: {lv_distance_sum / label_count}")
print(f"lv error rate: {lv_error_rate_sum / label_count}")
print(f"lv_correct_sum: {lv_correct_sum}, lv_incorrect_sum: {lv_incorrect_sum}, error_rate: {lv_incorrect_sum / (lv_correct_sum + lv_incorrect_sum)}")
print()
print(f"hm distance: {hm_distance_sum / label_count}")
print(f"hm error rate: {hm_error_rate_sum / label_count}")
print(f"hm_correct_sum: {hm_correct_sum}, hm_incorrect_sum: {hm_incorrect_sum}, error_rate: {hm_incorrect_sum / (hm_correct_sum + hm_incorrect_sum)}")
print()

## 7. Runtime Benchmark and TTS Generation


In [ ]:
path_json = str(ANSWERS_JSON)

with open(path_json, "r") as f:
  answers = json.load(f)

success = 0
time_predict = 0
time_extract = 0
time_tts = 0

for i, answer in enumerate(answers):
  path_image = str(IMAGES_DIR / answer["name"])
  image_name = os.path.basename(path_image)
  image = cv2.imread(path_image)
  print(f"[{i + 1}/{len(answers)}]: {image_name}")

  # Predict
  predict_start = time.perf_counter()
  results = model.predict(image, conf=0.3, imgsz=640, verbose=False)
  predict_time = time.perf_counter() - predict_start

  # Extract
  extract_start = time.perf_counter()
  data = extract_data(results[0], image)
  extract_time = time.perf_counter() - extract_start

  if data is not None:
    x_labels, bar_values, prefix, suffix, yb_value, yt_value = data
    description = create_description(x_labels, bar_values, prefix, suffix)

    tts_start = time.perf_counter()
    temp_audio_filename = "temp.wav"
    gTTS(description, lang="en").save(temp_audio_filename)
    tts_time = time.perf_counter() - tts_start

    success += 1
    time_predict += predict_time
    time_extract += extract_time
    time_tts += tts_time

# Summary statistics
time_predict_avg = time_predict / success
time_extract_avg = time_extract / success
time_tts_avg = time_tts / success
time_running_avg = time_predict_avg + time_extract_avg + time_tts_avg

print(f"Success: {success}/{len(answers)}")
print(f"Running time: {time_running_avg:.2f} s")
print(f"predict: {time_predict_avg:.3f} s ({100*(time_predict_avg/time_running_avg):.2f}%)")
print(f"extract: {time_extract_avg:.3f} s ({100*(time_extract_avg/time_running_avg):.2f}%)")
print(f"tts: {time_tts_avg:.3f} s ({100*(time_tts_avg/time_running_avg):.2f}%)")

## Required Files for Full Execution

Prepare the following files inside the repository:

### Required
- `models/yolov8s.yaml`
- `models/yolov8s-best-82.pt`
- `data/data_82.yaml`
- `data/train.txt`
- `data/val.txt`
- `data/images.json`
- chart images inside `data/images/`
- matching YOLO label files inside `data/labels/`

### Optional
- `models/yolov8s-best-91.pt`
- `data/data_91.yaml`

### Notes
- For a lightweight public repository, you can include only a small sample subset of images and labels.
- If you include only sample data, `train.txt`, `val.txt`, and `images.json` should match that sample subset exactly.
- To reproduce the full paper results, use the complete dataset and pretrained model files.
